In [1]:
import os
import json
from openai import OpenAI
from google.colab import userdata

  ==============================================
## Tool1 - Server Health Check Tool (stbd)
  ==============================================

In [2]:
def get_server_health(server_id: str) -> str:
    """Returns server CPU and memory usage."""
    print(f"-> TOOL: Checking health for {server_id}...")
    metrics = {
        # Server 1 - High CPU
        "payment-gateway-01": {"cpu": "99%", "memory": "35%", "status": "Warning"},
        # Server 2 - Healthy
        "db-SQL-02": {"cpu": "15%", "memory": "65%", "status": "Healthy"},
        # Server 3 - High Memory Leak
        "auth-SSO-03": {"cpu": "65%", "memory": "96%", "status": "Critical"},
        # Server 4 - Network, Dependency Failure
        "search-service-04": {"cpu": "15%", "memory": "25%", "status": "Error"},
        # Server 4 - No issues
        "frontend-node-05": {"cpu": "20%", "memory": "25%", "status": "Healthy"}
    }
    result = metrics.get(server_id, {"Error": "No Such Server Found"})
    return json.dumps(result)

 ====================================
## Tool2 - Fetch Logs Tool (stbd)
 ====================================

In [3]:
def fetch_recent_logs(server_id: str, lines: int = 5) -> str:
    """Returns the latest server logs."""
    print(f"-> TOOL: Fetching last {lines} log lines for {server_id}...")
    log_database = {
        "payment-gateway-01": [
            "[INFO] Request received /pay/v1",
            "[WARN] CPU threshold exceeded 90%",
            "[WARN] Thread pool depleted",
            "[CRITICAL] Process hung, not accepting new connections",
            "[ERROR] Timeout waiting for thread"
        ],
        "db-SQL-02": [
            "[INFO] Backup started",
            "[INFO] Backup completed successfully",
            "[INFO] User query executed in 10ms",
            "[INFO] Health check: OK",
            "[INFO] Replication sync active"
        ],
        "auth-SSO-03": [
            "[INFO] Token validated user_321",
            "[WARN] Garbage collection taking too long (>10s)",
            "[ERROR] java.lang.OutOfMemoryError: Java heap space",
            "[CRITICAL] Application crashed due to memory leak",
            "[INFO] Restarting context..."
        ],
        "search-service-04": [
            "[INFO] Indexing started",
            "[ERROR] Connection refused: elastic-cluster-main:9200",
            "[ERROR] Failed to write document ID 4442",
            "[CRITICAL] Dependency Unreachable: Search Engine is unresponsive",
            "[ERROR] Retrying in 30s..."
        ],
        "frontend-node-05": [
            "[INFO] GET /home 200 OK",
            "[INFO] GET /assets/logo.png 200 OK",
            "[INFO] GET /login 200 OK",
            "[INFO] GET /api/v1/status 200 OK",
            "[INFO] Health check passed"
        ]
    }
    default_logs = ["[INFO] System not found", "[INFO] Synthetic monitor heartbeat not received"]
    logs = log_database.get(server_id, default_logs)
    return json.dumps({"logs": logs[:lines]})

 ==============================================
## Tool3 - Server Auto-Restart Tool (stbd)
 ==============================================

In [4]:
def restart_service(server_id: str) -> str:
    """Restarts server by server_id"""
    print(f"-> TOOL: Restarting server: {server_id}")
    result = {
        "server_id": server_id,
        "status": "success",
        "message": "Server restart completed successfully."
    }
    return json.dumps(result)

 ==========================================================
## Tool4 - Human-in-the-loop Escalation Tool (stbd)
 ==========================================================

In [5]:
def escalate_to_engineer(summary: str) -> str:
    """Escalates to a human engineer"""
    print(f"-> TOOL: Escalating to a human engineer. Reason: {summary}")
    result = {
        "status": "escalated",
        "ticket_id": "INC-099",
        "assigned_to": "On-Call SRE Engineer"
    }
    return json.dumps(result)

 ================================================
## Agentic execution loop function-mapping
 ================================================

In [6]:
AVAILABLE_FUNCTIONS = {
    "get_server_health": get_server_health,
    "fetch_recent_logs": fetch_recent_logs,
    "restart_service": restart_service,
    "escalate_to_engineer": escalate_to_engineer,
}

 ========================================
## Define Tool Schemas
 ========================================

In [7]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_server_health",
            "description": "Checks a server’s current CPU and memory usage.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server, e.g., 'payment-server-01'"}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "fetch_recent_logs",
            "description": "Fetches recent server logs to diagnose errors.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server."},
                    "lines": {"type": "integer", "description": "Number of log lines to fetch."}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "restart_service",
            "description": "Restarts a server service when CPU usage is critical or server becomes unresponsive.",
            "parameters": {
                "type": "object",
                "properties": {
                    "server_id": {"type": "string", "description": "The ID of the server to restart."}
                },
                "required": ["server_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "escalate_to_engineer",
            "description": "Escalates the issue to a human engineer when automated fixes fail, or the error is complex (based on logs) or unknown.",
            "parameters": {
                "type": "object",
                "properties": {
                    "summary": {"type": "string", "description": "A brief summary of the findings (health status, log errors) and why you are escalating."}
                },
                "required": ["summary"]
            }
        }
    }
]

 ========================================
## Get LLM Key
 ========================================

In [8]:
client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))

 ========================================
## Initialize LLM / Workflow logic and
 ========================================

In [9]:
def run_it_agent(user_issue: str):
    print(f"\n--- New Incident: {user_issue} ---")
    messages = [
        {"role": "system", "content": "You are a IT Incident Responder. Investigate server issues. "
                                      "Restart the service if CPU or memory exceeds 90%. Escalate dependency errors a restart cannot fix. For example, if logs show critical dependency errors (like connection refused) then a restart won't fix the problem, escalate to an engineer."},
        {"role": "user", "content": user_issue}
    ]
    while True:
        print("\n[AI-Agent is thinking...]")
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools_schema,
            tool_choice="auto"
        )
        response_msg = response.choices[0].message
        messages.append(response_msg)
        if response_msg.tool_calls:
            for tool_call in response_msg.tool_calls:
                func_name = tool_call.function.name
                func_args = json.loads(tool_call.function.arguments)
                # Retrieve the tool
                function_to_call = AVAILABLE_FUNCTIONS.get(func_name)
                if function_to_call:
                    # Execute the tool
                    tool_output = function_to_call(**func_args)
                    # Append messages
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "name": func_name,
                        "content": tool_output
                    })
        else:
            print(f"\n[FINAL RESPONSE]: {response_msg.content}")
            break

 ========================================
## Execute Scenarios
 ========================================

In [10]:
# 1: When CPU > 90%. Expected output -> Automatic Restart
run_it_agent("payment-gateway-01 is showing extremely slow and is timing out.")
print("\n" + "="*50 + "\n")


--- New Incident: payment-gateway-01 is showing extremely slow and is timing out. ---

[AI-Agent is thinking...]
-> TOOL: Checking health for payment-gateway-01...
-> TOOL: Fetching last 50 log lines for payment-gateway-01...

[AI-Agent is thinking...]
-> TOOL: Restarting server: payment-gateway-01

[AI-Agent is thinking...]

[FINAL RESPONSE]: The payment-gateway-01 server was experiencing high CPU usage at 99%, causing it to become slow and time out. I have restarted the service to alleviate the CPU overload. Please monitor the server and let me know if the issue persists or if there are any other symptoms.




In [11]:
# 2: Warnings/Critical Errors in Logs. Expected output -> No restart or escalation required
run_it_agent("How is db-SQL-02 doing? Is something wrong?")
print("\n" + "="*50 + "\n")


--- New Incident: How is db-SQL-02 doing? Is something wrong? ---

[AI-Agent is thinking...]
-> TOOL: Checking health for db-SQL-02...
-> TOOL: Fetching last 20 log lines for db-SQL-02...

[AI-Agent is thinking...]

[FINAL RESPONSE]: The server db-SQL-02 is healthy with CPU usage at 15% and memory usage at 65%. The recent logs do not show any errors or warnings; everything appears normal and operational. There is nothing wrong with this server at the moment.




In [12]:
# 3: When System Memory > 90% OR OutOfMemoryError seen in logs. Expected output -> Automatic Restart
run_it_agent("Multiple poduction users are reporting login failures, check auth-SSO-03")
print("\n" + "="*50 + "\n")


--- New Incident: Multiple poduction users are reporting login failures, check auth-SSO-03 ---

[AI-Agent is thinking...]
-> TOOL: Checking health for auth-SSO-03...
-> TOOL: Fetching last 50 log lines for auth-SSO-03...

[AI-Agent is thinking...]
-> TOOL: Restarting server: auth-SSO-03

[AI-Agent is thinking...]

[FINAL RESPONSE]: The auth-SSO-03 server had critical memory usage and logs indicated an OutOfMemoryError causing the application to crash. I have restarted the service to resolve the issue. Please monitor the login functionality and let me know if the problem persists or if there are any new errors.




In [13]:
# 4: When 'Connection Refused' seen in logs. Expected output -> Escalation
run_it_agent("Search is just not working. Please check search-service-04.")
print("\n" + "="*50 + "\n")


--- New Incident: Search is just not working. Please check search-service-04. ---

[AI-Agent is thinking...]
-> TOOL: Checking health for search-service-04...
-> TOOL: Fetching last 50 log lines for search-service-04...

[AI-Agent is thinking...]
-> TOOL: Escalating to a human engineer. Reason: Search service (search-service-04) shows normal CPU (15%) and memory usage (25%) but has critical errors in logs indicating dependency errors: connection refused to elastic-cluster-main on port 9200, search engine unresponsive. Restart won't fix as it is a dependency failure. Escalating to an engineer for investigation.

[AI-Agent is thinking...]

[FINAL RESPONSE]: The search service on search-service-04 is encountering critical dependency errors related to the connection refusal to the elastic-cluster-main, causing the search engine to be unresponsive. CPU and memory usage are normal, so restarting the service won't fix the dependency issue. I have escalated this incident to the on-call SRE en

In [15]:
# 5: When CPU/Memory metrics and Logs are normal. Expected output -> No restart or escalation required
run_it_agent("Check frontend-node-05 now.")
print("\n" + "="*50 + "\n")


--- New Incident: Check frontend-node-05 now. ---

[AI-Agent is thinking...]
-> TOOL: Checking health for frontend-node-05...
-> TOOL: Fetching last 50 log lines for frontend-node-05...

[AI-Agent is thinking...]

[FINAL RESPONSE]: The server frontend-node-05 is healthy with CPU usage at 20% and memory usage at 25%. The recent logs do not show any errors, only info level messages indicating normal operations. No restart or escalation is needed at this time. Let me know if you want me to check anything else.




In [16]:
# 6: When Server is not found. Expected output -> Server not found.
run_it_agent("Run a check on backend-node-06 please.")
print("\n" + "="*50 + "\n")


--- New Incident: Run a check on backend-node-06 please. ---

[AI-Agent is thinking...]
-> TOOL: Checking health for backend-node-06...
-> TOOL: Fetching last 50 log lines for backend-node-06...

[AI-Agent is thinking...]

[FINAL RESPONSE]: The server "backend-node-06" does not exist or cannot be found. Additionally, recent logs for this server indicate "System not found" and lack of synthetic monitor heartbeat.

Please confirm the server ID or provide another server ID for investigation.


